In [0]:
CATALOG = "worldbank_ai"
BRONZE_SCHEMA = "bronze"

COUNTRIES_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.countries_raw"

WORLD_BANK_API_BASE_URL = "https://api.worldbank.org/v2"

print(f"Target table: {COUNTRIES_TABLE}")
print(f"API base URL: {WORLD_BANK_API_BASE_URL}")

Target table: worldbank_ai.bronze.countries_raw
API base URL: https://api.worldbank.org/v2


In [0]:
# Imports

import requests
import time

from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql import types as T

print("Imports loaded.")

Imports loaded.


In [0]:
# API request helper

# COMMAND ----------

def get_json_with_retry(
    url,
    params=None,
    max_retries=4,
    timeout=30
):
    """
    Send a GET request and return parsed JSON.

    Retries temporary failures such as:
    - HTTP 429 rate limiting
    - HTTP 5xx server errors

    Raises an exception for permanent failures.
    """

    retryable_status_codes = {
        429, 500, 502, 503, 504
    }

    for attempt in range(1, max_retries + 1):

        try:
            response = requests.get(
                url,
                params=params,
                timeout=timeout
            )

            if response.status_code == 200:
                return response.json()

            if response.status_code in retryable_status_codes:

                wait_seconds = min(2 ** attempt, 30)

                print(
                    f"Temporary API error "
                    f"{response.status_code}. "
                    f"Retrying in {wait_seconds}s..."
                )

                time.sleep(wait_seconds)
                continue

            response.raise_for_status()

        except requests.RequestException as exc:

            if attempt == max_retries:
                raise RuntimeError(
                    f"World Bank API request failed "
                    f"after {max_retries} attempts."
                ) from exc

            wait_seconds = min(2 ** attempt, 30)

            print(
                f"Request error: {exc}. "
                f"Retrying in {wait_seconds}s..."
            )

            time.sleep(wait_seconds)

    raise RuntimeError(
        "World Bank API request failed unexpectedly."
    )

In [0]:
# Test the World Bank API


COUNTRIES_ENDPOINT = (
    f"{WORLD_BANK_API_BASE_URL}/country"
)

test_params = {
    "format": "json",
    "per_page": 5,
    "page": 1
}

test_response = get_json_with_retry(
    COUNTRIES_ENDPOINT,
    params=test_params
)

print(type(test_response))
print(f"Top-level elements: {len(test_response)}")

<class 'list'>
Top-level elements: 2


In [0]:
# Inspect API metadata

api_metadata = test_response[0]

print("API response metadata")
print("-" * 60)

for key, value in api_metadata.items():
    print(f"{key}: {value}")

API response metadata
------------------------------------------------------------
page: 1
pages: 59
per_page: 5
total: 295


In [0]:
# Inspect one source record


sample_records = test_response[1]

if not sample_records:
    raise RuntimeError(
        "World Bank API returned no country records."
    )

sample_record = sample_records[0]

print("Sample World Bank country record")
print("-" * 60)

for key, value in sample_record.items():
    print(f"{key}: {value}")

Sample World Bank country record
------------------------------------------------------------
id: ABW
iso2Code: AW
name: Aruba
region: {'id': 'LCN', 'iso2code': 'ZJ', 'value': 'Latin America & Caribbean '}
adminregion: {'id': '', 'iso2code': '', 'value': ''}
incomeLevel: {'id': 'HIC', 'iso2code': 'XD', 'value': 'High income'}
lendingType: {'id': 'LNX', 'iso2code': 'XX', 'value': 'Not classified'}
capitalCity: Oranjestad
longitude: -70.0167
latitude: 12.5167


In [0]:
# Fetch all pages

def fetch_all_countries():

    first_page = get_json_with_retry(
        COUNTRIES_ENDPOINT,
        params={
            "format": "json",
            "per_page": 100,
            "page": 1
        }
    )

    if (
        not isinstance(first_page, list)
        or len(first_page) < 2
    ):
        raise ValueError(
            "Unexpected World Bank API response structure."
        )

    metadata = first_page[0]
    records = first_page[1] or []

    total_pages = int(metadata["pages"])

    print(f"Total API pages: {total_pages}")
    print(f"Expected source records: {metadata['total']}")

    all_records = list(records)

    for page in range(2, total_pages + 1):

        print(
            f"Fetching page {page}/{total_pages}"
        )

        response = get_json_with_retry(
            COUNTRIES_ENDPOINT,
            params={
                "format": "json",
                "per_page": 100,
                "page": page
            }
        )

        if (
            not isinstance(response, list)
            or len(response) < 2
        ):
            raise ValueError(
                f"Unexpected response on page {page}."
            )

        page_records = response[1] or []

        all_records.extend(page_records)

    return all_records, metadata

In [0]:

country_records, source_metadata = fetch_all_countries()

print()
print(f"Records downloaded: {len(country_records)}")

Total API pages: 3
Expected source records: 295
Fetching page 2/3
Fetching page 3/3

Records downloaded: 295


In [0]:
# Validate source record count

expected_count = int(source_metadata["total"])
actual_count = len(country_records)

print(f"API expected: {expected_count}")
print(f"Downloaded:   {actual_count}")

if actual_count != expected_count:
    raise RuntimeError(
        f"Record-count mismatch. "
        f"API reports {expected_count}, "
        f"but ingestion retrieved {actual_count}."
    )

print("Source record-count validation passed.")

API expected: 295
Downloaded:   295
Source record-count validation passed.


In [0]:
# Define the Bronze schema

bronze_schema = T.StructType([
    T.StructField("country_id", T.StringType(), True),
    T.StructField("iso2_code", T.StringType(), True),
    T.StructField("country_name", T.StringType(), True),

    T.StructField("region_id", T.StringType(), True),
    T.StructField("region_code", T.StringType(), True),
    T.StructField("region_name", T.StringType(), True),

    T.StructField("admin_region_id", T.StringType(), True),
    T.StructField("admin_region_code", T.StringType(), True),
    T.StructField("admin_region_name", T.StringType(), True),

    T.StructField("income_level_id", T.StringType(), True),
    T.StructField("income_level_code", T.StringType(), True),
    T.StructField("income_level_name", T.StringType(), True),

    T.StructField("lending_type_id", T.StringType(), True),
    T.StructField("lending_type_code", T.StringType(), True),
    T.StructField("lending_type_name", T.StringType(), True),

    T.StructField("capital_city", T.StringType(), True),
    T.StructField("longitude", T.StringType(), True),
    T.StructField("latitude", T.StringType(), True),

    T.StructField("source_system", T.StringType(), False),
    T.StructField("source_endpoint", T.StringType(), False),
    T.StructField("ingested_at", T.TimestampType(), False)
])

In [0]:
# Convert API records to Bronze records


ingestion_timestamp = datetime.now(timezone.utc)

bronze_records = []

for record in country_records:

    region = record.get("region") or {}
    admin_region = record.get("adminregion") or {}
    income_level = record.get("incomeLevel") or {}
    lending_type = record.get("lendingType") or {}

    bronze_records.append({
        "country_id": record.get("id"),
        "iso2_code": record.get("iso2Code"),
        "country_name": record.get("name"),

        "region_id": region.get("id"),
        "region_code": region.get("iso2code"),
        "region_name": region.get("value"),

        "admin_region_id": admin_region.get("id"),
        "admin_region_code": admin_region.get("iso2code"),
        "admin_region_name": admin_region.get("value"),

        "income_level_id": income_level.get("id"),
        "income_level_code": income_level.get("iso2code"),
        "income_level_name": income_level.get("value"),

        "lending_type_id": lending_type.get("id"),
        "lending_type_code": lending_type.get("iso2code"),
        "lending_type_name": lending_type.get("value"),

        "capital_city": record.get("capitalCity"),
        "longitude": record.get("longitude"),
        "latitude": record.get("latitude"),

        "source_system": "World Bank Indicators API",
        "source_endpoint": COUNTRIES_ENDPOINT,
        "ingested_at": ingestion_timestamp
    })

In [0]:
# Create Spark DataFrame


countries_bronze_df = spark.createDataFrame(
    bronze_records,
    schema=bronze_schema
)

print(
    f"Bronze DataFrame rows: "
    f"{countries_bronze_df.count()}"
)

display(countries_bronze_df.limit(20))

Bronze DataFrame rows: 295


country_id,iso2_code,country_name,region_id,region_code,region_name,admin_region_id,admin_region_code,admin_region_name,income_level_id,income_level_code,income_level_name,lending_type_id,lending_type_code,lending_type_name,capital_city,longitude,latitude,source_system,source_endpoint,ingested_at
ABW,AW,Aruba,LCN,ZJ,Latin America & Caribbean,,,,HIC,XD,High income,LNX,XX,Not classified,Oranjestad,-70.0167,12.5167,World Bank Indicators API,https://api.worldbank.org/v2/country,2026-09-24T23:35:12.039Z
AFE,ZH,Africa Eastern and Southern,NA,NA,Aggregates,,,,NA,NA,Aggregates,,,Aggregates,,,,World Bank Indicators API,https://api.worldbank.org/v2/country,2026-09-24T23:35:12.039Z
AFG,AF,Afghanistan,MEA,ZQ,"Middle East, North Africa, Afghanistan & Pakistan",MNA,XQ,"Middle East, North Africa, Afghanistan & Pakistan (excluding high income)",LIC,XM,Low income,IDX,XI,IDA,Kabul,69.1761,34.5228,World Bank Indicators API,https://api.worldbank.org/v2/country,2026-09-24T23:35:12.039Z
AFR,A9,Africa,NA,NA,Aggregates,,,,NA,NA,Aggregates,,,Aggregates,,,,World Bank Indicators API,https://api.worldbank.org/v2/country,2026-09-24T23:35:12.039Z
AFW,ZI,Africa Western and Central,NA,NA,Aggregates,,,,NA,NA,Aggregates,,,Aggregates,,,,World Bank Indicators API,https://api.worldbank.org/v2/country,2026-09-24T23:35:12.039Z
AGO,AO,Angola,SSF,ZG,Sub-Saharan Africa,SSA,ZF,Sub-Saharan Africa (excluding high income),LMC,XN,Lower middle income,IBD,XF,IBRD,Luanda,13.242,-8.81155,World Bank Indicators API,https://api.worldbank.org/v2/country,2026-09-24T23:35:12.039Z
ALB,AL,Albania,ECS,Z7,Europe & Central Asia,ECA,7E,Europe & Central Asia (excluding high income),UMC,XT,Upper middle income,IBD,XF,IBRD,Tirane,19.8172,41.3317,World Bank Indicators API,https://api.worldbank.org/v2/country,2026-09-24T23:35:12.039Z
AND,AD,Andorra,ECS,Z7,Europe & Central Asia,,,,HIC,XD,High income,LNX,XX,Not classified,Andorra la Vella,1.5218,42.5075,World Bank Indicators API,https://api.worldbank.org/v2/country,2026-09-24T23:35:12.039Z
ARB,1A,Arab World,NA,NA,Aggregates,,,,NA,NA,Aggregates,,,Aggregates,,,,World Bank Indicators API,https://api.worldbank.org/v2/country,2026-09-24T23:35:12.039Z
ARE,AE,United Arab Emirates,MEA,ZQ,"Middle East, North Africa, Afghanistan & Pakistan",,,,HIC,XD,High income,LNX,XX,Not classified,Abu Dhabi,54.3705,24.4764,World Bank Indicators API,https://api.worldbank.org/v2/country,2026-09-24T23:35:12.039Z


In [0]:
# Basic Bronze validation

total_rows = countries_bronze_df.count()

null_id_rows = (
    countries_bronze_df
    .filter(F.col("country_id").isNull())
    .count()
)

duplicate_ids = (
    countries_bronze_df
    .groupBy("country_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Total rows: {total_rows}")
print(f"Null country IDs: {null_id_rows}")
print(f"Duplicate country IDs: {duplicate_ids}")

if total_rows == 0:
    raise RuntimeError(
        "No country records available for Bronze write."
    )

if null_id_rows > 0:
    raise RuntimeError(
        "Country metadata contains null source IDs."
    )

if duplicate_ids > 0:
    raise RuntimeError(
        "Duplicate source country IDs detected."
    )

print("Bronze validation passed.")

Total rows: 295
Null country IDs: 0
Duplicate country IDs: 0
Bronze validation passed.


In [0]:
# Write the Bronze Delta table


(
    countries_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(COUNTRIES_TABLE)
)

print(
    f"Bronze table written successfully: "
    f"{COUNTRIES_TABLE}"
)

Bronze table written successfully: worldbank_ai.bronze.countries_raw


In [0]:
# Read it back

saved_df = spark.table(COUNTRIES_TABLE)

saved_count = saved_df.count()

print(f"Source records: {actual_count}")
print(f"Saved records:  {saved_count}")

if saved_count != actual_count:
    raise RuntimeError(
        "Bronze write validation failed: "
        "saved row count differs from source."
    )

print("Bronze write validation passed.")

display(saved_df.limit(20))

Source records: 295
Saved records:  295
Bronze write validation passed.


country_id,iso2_code,country_name,region_id,region_code,region_name,admin_region_id,admin_region_code,admin_region_name,income_level_id,income_level_code,income_level_name,lending_type_id,lending_type_code,lending_type_name,capital_city,longitude,latitude,source_system,source_endpoint,ingested_at
ABW,AW,Aruba,LCN,ZJ,Latin America & Caribbean,,,,HIC,XD,High income,LNX,XX,Not classified,Oranjestad,-70.0167,12.5167,World Bank Indicators API,https://api.worldbank.org/v2/country,2026-09-24T23:35:12.039Z
AFE,ZH,Africa Eastern and Southern,NA,NA,Aggregates,,,,NA,NA,Aggregates,,,Aggregates,,,,World Bank Indicators API,https://api.worldbank.org/v2/country,2026-09-24T23:35:12.039Z
AFG,AF,Afghanistan,MEA,ZQ,"Middle East, North Africa, Afghanistan & Pakistan",MNA,XQ,"Middle East, North Africa, Afghanistan & Pakistan (excluding high income)",LIC,XM,Low income,IDX,XI,IDA,Kabul,69.1761,34.5228,World Bank Indicators API,https://api.worldbank.org/v2/country,2026-09-24T23:35:12.039Z
AFR,A9,Africa,NA,NA,Aggregates,,,,NA,NA,Aggregates,,,Aggregates,,,,World Bank Indicators API,https://api.worldbank.org/v2/country,2026-09-24T23:35:12.039Z
AFW,ZI,Africa Western and Central,NA,NA,Aggregates,,,,NA,NA,Aggregates,,,Aggregates,,,,World Bank Indicators API,https://api.worldbank.org/v2/country,2026-09-24T23:35:12.039Z
AGO,AO,Angola,SSF,ZG,Sub-Saharan Africa,SSA,ZF,Sub-Saharan Africa (excluding high income),LMC,XN,Lower middle income,IBD,XF,IBRD,Luanda,13.242,-8.81155,World Bank Indicators API,https://api.worldbank.org/v2/country,2026-09-24T23:35:12.039Z
ALB,AL,Albania,ECS,Z7,Europe & Central Asia,ECA,7E,Europe & Central Asia (excluding high income),UMC,XT,Upper middle income,IBD,XF,IBRD,Tirane,19.8172,41.3317,World Bank Indicators API,https://api.worldbank.org/v2/country,2026-09-24T23:35:12.039Z
AND,AD,Andorra,ECS,Z7,Europe & Central Asia,,,,HIC,XD,High income,LNX,XX,Not classified,Andorra la Vella,1.5218,42.5075,World Bank Indicators API,https://api.worldbank.org/v2/country,2026-09-24T23:35:12.039Z
ARB,1A,Arab World,NA,NA,Aggregates,,,,NA,NA,Aggregates,,,Aggregates,,,,World Bank Indicators API,https://api.worldbank.org/v2/country,2026-09-24T23:35:12.039Z
ARE,AE,United Arab Emirates,MEA,ZQ,"Middle East, North Africa, Afghanistan & Pakistan",,,,HIC,XD,High income,LNX,XX,Not classified,Abu Dhabi,54.3705,24.4764,World Bank Indicators API,https://api.worldbank.org/v2/country,2026-09-24T23:35:12.039Z


In [0]:
display(
    saved_df
    .groupBy(
        "region_id",
        "region_name"
    )
    .count()
    .orderBy(F.desc("count"))
)

region_id,region_name,count
NA,Aggregates,78
ECS,Europe & Central Asia,58
SSF,Sub-Saharan Africa,48
LCN,Latin America & Caribbean,42
EAS,East Asia & Pacific,37
MEA,"Middle East, North Africa, Afghanistan & Pakistan",23
SAS,South Asia,6
NAC,North America,3


In [0]:
# Inspect regions

display(
    saved_df
    .groupBy(
        "region_id",
        "region_name"
    )
    .count()
    .orderBy(F.desc("count"))
)

region_id,region_name,count
NA,Aggregates,78
ECS,Europe & Central Asia,58
SSF,Sub-Saharan Africa,48
LCN,Latin America & Caribbean,42
EAS,East Asia & Pacific,37
MEA,"Middle East, North Africa, Afghanistan & Pakistan",23
SAS,South Asia,6
NAC,North America,3


In [0]:
# Final summary


print("=" * 65)
print("WORLD BANK COUNTRY METADATA INGESTION")
print("=" * 65)

print(f"Source: World Bank Indicators API")
print(f"Endpoint: {COUNTRIES_ENDPOINT}")
print(f"Records retrieved: {actual_count}")
print(f"Records stored: {saved_count}")
print(f"Target: {COUNTRIES_TABLE}")
print("Format: Delta")
print("Layer: Bronze")
print("Status: SUCCESS")

WORLD BANK COUNTRY METADATA INGESTION
Source: World Bank Indicators API
Endpoint: https://api.worldbank.org/v2/country
Records retrieved: 295
Records stored: 295
Target: worldbank_ai.bronze.countries_raw
Format: Delta
Layer: Bronze
Status: SUCCESS
